# 03 · Class / category mean embeddings

**What it does.** Turns per-exemplar embeddings into per-class sums and means, which notebook 05 pools into CDI
categories.
- **ImageNet:** whole images (train + val, 1,331,167), embedded by `data_prep/run_embeddings_imagenet_lvis.sh`.
- **LVIS:** each labeled box cropped and embedded by `data_prep/run_lvis_crop_embeddings.sh` (928,726 boxes; the 473
  under 1 px cannot be cropped). Whole-image LVIS embeddings exist but are not used: an LVIS image is a scene.
- **Each exemplar's embedding is L2-normalized before averaging**, the same as BV, VEDB and THINGS. Checkpoints:
  CLIP ViT-B/32, DINOv3 ViT-B/16 CLS (the lab HF cache BV used).

**Runtime:** recomputing reads ~1M small files on NFS (~45–50 min per model), so by default the notebook **loads**
the saved means and checks them; set `RECOMPUTE = True` to rebuild.

**Outputs** (`outputs/mapping/`): `{imagenet_class,lvis_category}_embeddings_{clip,dinov3}.npz`, and for LVIS
`lvis_crop_size_by_category_exclusive104.csv`.

In [1]:
# Standard setup shared by every notebook (each notebook is self-contained).
import json
import os
import random
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist, squareform
from scipy.stats import false_discovery_control, pearsonr, rankdata, spearmanr, wilcoxon
from IPython.display import Image, Markdown, display

plt.rcParams["font.family"] = ["DejaVu Sans", "sans-serif"]
pd.set_option("display.max_colwidth", 120)


def show_images(paths, width=950):
    """Display saved figures (the files this notebook writes)."""
    for p in map(Path, paths):
        if p.exists():
            display(Markdown(f"`{p.relative_to(ROOT)}`")); display(Image(filename=str(p), width=width))
        else:
            display(Markdown(f"*Missing figure:* `{p}`"))

In [2]:
# Paths. Run this notebook from analysis/comparison-datasets/notebooks/ (Jupyter's default working directory).
ROOT = Path.cwd().resolve().parent                     # analysis/comparison-datasets
assert (ROOT / "notebooks").is_dir(), "open this notebook from analysis/comparison-datasets/notebooks/"
REPO = ROOT.parents[1]
DATA_DIR = REPO / "data"
MANUSCRIPT_DIR = REPO / "analysis" / "manuscript-2026"
VEDB_DIR = REPO / "analysis" / "vedb-1612"
VOCAB_DIR = DATA_DIR / "vocabulary_lists"
VALID129_TXT = VOCAB_DIR / "included_categories_valid129.txt"

# Inputs written by other pipelines (read only).
EXEMPLAR_EMBED_ROOT = Path(os.environ.get("BV_EXEMPLAR_EMBED_ROOT",
                                          str(MANUSCRIPT_DIR / "exemplar_set_embeddings"))).expanduser()
BV_THINGS_VALID129 = EXEMPLAR_EMBED_ROOT / "valid129"       # manuscript Stage 0: BV + THINGS tables
VEDB_VALID129 = VEDB_DIR / "exemplar_set_embeddings" / "valid129"   # vedb-1612 Stage 0 (files use the "bv_" prefix)

# External data on ccn2.
IMAGENET_DIR = Path("/data2/j7yang/imagenet-1k/extracted")
IMAGENET_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings")
LVIS_ROOT = Path("/data2/j7yang/lvis")
LVIS_CROP_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings")

# Outputs of this folder.
OUT = ROOT / "outputs"
MAPPING_OUT = OUT / "mapping"
MONTAGE_OUT = OUT / "montages"
EXEMPLAR_SETS = OUT / "exemplar_sets"
MODELS = ("clip", "dinov3")

## Parameters

In [3]:
DATASET = "lvis"
RECOMPUTE = False   # True: rebuild the .npz files (slow)
assert DATASET in ('imagenet', 'lvis'), "DATASET must be one of ('imagenet', 'lvis')"

## The embedding run (shell, GPU)

In [4]:
script = "run_embeddings_imagenet_lvis.sh" if DATASET == "imagenet" else "run_lvis_crop_embeddings.sh"
print((ROOT / "data_prep" / script).read_text())

#!/bin/bash
# CLIP + DINOv3 embeddings of LVIS v1 ground-truth box crops (train + val annotations),
# for every LVIS category mapped to a CDI word by WordNet hyponym pooling
# (notebooks/01_cdi_mapping.ipynb with DATASET = "lvis" -> outputs/mapping/lvis_crop_manifest.csv).
#
# Human-annotated boxes: no detector, no CLIP filter, no box-size filter. The
# manifest's xmin..ymax columns make create_image_embeddings.py crop each box
# before embedding; output files are <out>/<model_tag>/lvis_<split>_<ann_id>.npy.
#
# The whole-image LVIS embeddings from run_embeddings_imagenet_lvis.sh are NOT used
# for category analyses (an LVIS image is a multi-object scene) -- crops go to a
# separate crop_embeddings/ tree.
#
# HF_HOME points at the lab cache that holds the gated DINOv3 weights (same as
# image-embedding/dispatch_cdi_crop_pipeline.sh), so the checkpoints match BV/VEDB/ImageNet.
# Shared node: fractional-GPU Ray workers on two GPUs that were free at launch
# (re-check nvidia-smi before re

## ImageNet class means

In [5]:
IMAGENET_MODEL_DIRS = {
    "clip": "openai_clip-vit-base-patch32",
    "dinov3": "facebook_dinov3-vitb16-pretrain-lvd1689m",  # CLS / pooler_output, as for THINGS + BV
}


def list_stems_by_wnid() -> dict:
    """{wnid: ['train_<stem>', 'val_<stem>', ...]} -- embedding ids are <split>_<image stem>."""
    out = defaultdict(list)
    for split in ("train", "val"):
        with os.scandir(IMAGENET_DIR / split) as it:
            for e in it:
                if e.name.endswith(".JPEG"):
                    stem = e.name[:-5]
                    out[stem.rsplit("_", 1)[1]].append(f"{split}_{stem}")
    return out


def class_sum(embed_dir: Path, stems: list) -> tuple:
    total, n, missing = None, 0, 0
    for s in stems:
        p = embed_dir / f"{s}.npy"
        try:
            v = np.load(p).astype(np.float64).ravel()
        except FileNotFoundError:
            missing += 1
            continue
        nrm = np.linalg.norm(v)
        if nrm > 0:
            v = v / nrm
        total = v if total is None else total + v
        n += 1
    return total, n, missing


def run_imagenet_means(models=tuple(IMAGENET_MODEL_DIRS), workers: int = 16, out_dir: Path = MAPPING_OUT):
    """Sum L2-normalized per-image embeddings per ImageNet class; write one .npz per model (slow: ~50 min/model on NFS)."""

    stems = list_stems_by_wnid()
    wnids = sorted(stems)
    assert len(wnids) == 1000, len(wnids)
    print(f"{sum(map(len, stems.values()))} images over {len(wnids)} classes", flush=True)

    for model in models:
        embed_dir = IMAGENET_EMBED_ROOT / IMAGENET_MODEL_DIRS[model]
        with ThreadPoolExecutor(workers) as ex:
            results = []
            for i, r in enumerate(ex.map(lambda w: class_sum(embed_dir, stems[w]), wnids)):
                results.append(r)
                if (i + 1) % 100 == 0:
                    print(f"[{model}] {i + 1}/1000 classes", flush=True)
        sums = np.stack([r[0] for r in results])
        counts = np.array([r[1] for r in results])
        missing = sum(r[2] for r in results)
        out = out_dir / f"imagenet_class_embeddings_{model}.npz"
        np.savez(out, wnids=np.array(wnids), counts=counts, sums=sums, means=sums / counts[:, None],
                 embed_dir=str(embed_dir))
        print(f"[{model}] wrote {out}: {counts.sum()} images, {missing} missing .npy", flush=True)

    return out_dir

## LVIS category means

In [6]:
LVIS_MODEL_DIRS = {
    "clip": "openai_clip-vit-base-patch32",
    "dinov3": "facebook_dinov3-vitb16-pretrain-lvd1689m",  # CLS / pooler_output, as for THINGS + BV
}


def category_sum(embed_dir: Path, crop_ids: list) -> tuple:
    total, n, missing = None, 0, 0
    for cid in crop_ids:
        try:
            v = np.load(embed_dir / f"{cid}.npy").astype(np.float64).ravel()
        except FileNotFoundError:
            missing += 1
            continue
        nrm = np.linalg.norm(v)
        if nrm > 0:
            v = v / nrm
        total = v if total is None else total + v
        n += 1
    return total, n, missing


def run_lvis_means(models=tuple(LVIS_MODEL_DIRS), workers: int = 16, out_dir: Path = MAPPING_OUT):
    """Sum L2-normalized per-crop embeddings per LVIS category; write one .npz per model (slow: ~45 min/model on NFS)."""

    meta = pd.read_csv(out_dir / "lvis_crop_metadata.csv.gz", usecols=["crop_id", "category_id", "lvis_name"])
    groups = meta.groupby("category_id")
    cat_ids = sorted(groups.groups)
    names = groups["lvis_name"].first().loc[cat_ids].to_numpy()
    crops = {c: g["crop_id"].tolist() for c, g in groups}
    print(f"{len(meta)} crops over {len(cat_ids)} LVIS categories", flush=True)

    for model in models:
        embed_dir = LVIS_CROP_EMBED_ROOT / LVIS_MODEL_DIRS[model]
        with ThreadPoolExecutor(workers) as ex:
            results = list(ex.map(lambda c: category_sum(embed_dir, crops[c]), cat_ids))
        dim = next(r[0].shape[0] for r in results if r[0] is not None)
        sums = np.stack([r[0] if r[0] is not None else np.zeros(dim) for r in results])
        counts = np.array([r[1] for r in results])
        missing = sum(r[2] for r in results)
        with np.errstate(invalid="ignore", divide="ignore"):
            means = sums / counts[:, None]
        out = out_dir / f"lvis_category_embeddings_{model}.npz"
        np.savez(out, category_ids=np.array(cat_ids), names=names, counts=counts, sums=sums, means=means,
                 embed_dir=str(embed_dir))
        print(f"[{model}] wrote {out}: {counts.sum()} crops, {missing} missing .npy", flush=True)
        if missing:
            print(f"[{model}] WARNING: {missing} crops have no embedding (zero-size after clamping, or run incomplete)")

    return out_dir

In [7]:
if RECOMPUTE:
    run_imagenet_means() if DATASET == "imagenet" else run_lvis_means()

## Check the saved means
Counts per class, classes with no exemplars, and that every mean is finite and averaged from unit vectors (norm ≤ 1).

In [8]:
name = "imagenet_class_embeddings" if DATASET == "imagenet" else "lvis_category_embeddings"
rows = []
for model in MODELS:
    z = np.load(MAPPING_OUT / f"{name}_{model}.npz")
    norms = np.linalg.norm(z["means"], axis=1)
    rows.append({"model": model, "classes": len(z["counts"]), "exemplars": int(z["counts"].sum()),
                 "classes with 0 exemplars": int((z["counts"] == 0).sum()),
                 "all means finite": bool(np.isfinite(z["means"]).all()),
                 "max mean norm (<=1)": float(norms.max()), "embedding dir": str(z["embed_dir"])})
pd.DataFrame(rows)

,model,classes,exemplars,classes with 0 exemplars,all means finite,max mean norm (<=1),embedding dir
0,clip,559,928253,0,True,1.0,/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings/openai_clip-vit-base-patch32
1,dinov3,559,928253,0,True,1.0,/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings/facebook_dinov3-vitb16-pretrain-lvd1689m


## LVIS only: crop sizes behind each analyzed CDI category
Short side of each box in pixels. No size filter was applied, so this is what the means average over.

In [9]:
def lvis_crop_size_summary():
    man = pd.read_csv(MAPPING_OUT / "lvis_crop_manifest.csv").set_index("image_id")
    pools = pd.read_csv(EXEMPLAR_SETS / "lvis_exclusive104" / "lvis_category_classes.csv")
    meta = pd.read_csv(MAPPING_OUT / "lvis_crop_metadata.csv.gz", usecols=["crop_id", "category_id"])
    short = np.minimum(man["xmax"] - man["xmin"], man["ymax"] - man["ymin"])
    meta["short"] = meta["crop_id"].map(short)
    meta = meta[meta["short"] >= 1]
    d = meta.merge(pools[["cdi_category", "lvis_category_id"]], left_on="category_id", right_on="lvis_category_id")
    g = d.groupby("cdi_category")["short"]
    t = pd.DataFrame({
        "n": g.size(),
        "median_short_px": g.median(),
        "pct_lt16": g.apply(lambda s: (s < 16).mean() * 100),
        "pct_lt32": g.apply(lambda s: (s < 32).mean() * 100),
    }).round(1).sort_values("median_short_px")
    out = MAPPING_OUT / "lvis_crop_size_by_category_exclusive104.csv"
    t.to_csv(out)
    print(f"{len(d)} crops: median short side {d['short'].median():.0f} px, "
          f"<16 px {(d['short'] < 16).mean():.0%}, <32 px {(d['short'] < 32).mean():.0%} -> {out}")

In [10]:
if DATASET == "lvis":
    lvis_crop_size_summary()
    display(pd.read_csv(MAPPING_OUT / "lvis_crop_size_by_category_exclusive104.csv").head(15))

603785 crops: median short side 24 px, <16 px 35%, <32 px 59% -> /home/j7yang/babyview-projects/vss2026/object-detection/analysis/comparison-datasets/outputs/mapping/lvis_crop_size_by_category_exclusive104.csv


,cdi_category,n,median_short_px,pct_lt16,pct_lt32
0,glasses,14396,8.7,70.8,89.8
1,book,40824,9.5,72.3,90.5
2,pumpkin,1614,10.3,66.1,85.8
3,belt,4350,11.3,66.2,90.5
4,watch,3271,12.3,63.0,87.7
5,hat,9712,12.4,59.4,81.1
6,ball,5482,12.5,61.8,85.8
7,bird,20964,13.2,56.2,74.9
8,shoe,16258,13.5,57.6,86.1
9,sock,8266,14.4,54.8,85.7


### Review checklist

- Is every exemplar L2-normalized before averaging?
- Do the totals match the embedding runs (ImageNet 1,331,167; LVIS 928,253 = 928,726 − 473)?
- LVIS: how much do tiny crops (median short side 24 px) dominate each category?